# CSIRO Image2Biomass: Multi-Backbone Super-Ensemble Inference

This notebook implements **Multi-Backbone Super-Ensemble Inference & Submission** combining predictions from both **DINOv3 ViT** and **ConvNeXt-V2 Large** 5-fold models.

### 🏆 Why ViT + ConvNeXt Multi-Backbone Ensembling Wins
1. **Inductive Complementarity**: ViT uses global self-attention across image patch tokens; ConvNeXt-V2 uses 7×7 depthwise convolutions with Global Response Normalization (GRN). Ensembling both cancels out individual structural errors and was the cornerstone of the top-3 solutions.
2. **Automatic Architecture Recognition**: Checks checkpoint layer shapes (768 vs 1536 channels) and automatically instantiates the exact corresponding backbone (`vit_base_patch16_dinov3_qkvb` or `convnextv2_large`).
3. **Dual-Stream 1:1 Square Splitting**: $2000 \times 1000 \to$ Left & Right $1000 \times 1000$ views resized to $512 \times 512$.
4. **Cross-View Multi-Head Self-Attention**: Spatial interaction across the seam for both ViT and ConvNeXt.
5. **Test-Time Augmentation (TTA)**: Evaluates normal views and horizontally flipped mirrored views.
6. **Soft Physical Calibration & Fringe Expansion**: Calibrates mass conservation and expands dead thatch dynamic range.
7. **Multi-Output Submissions**: Outputs `submission.csv` (the blended super-ensemble), plus `submission_vit.csv` and `submission_convnext.csv` for fine-grained validation.

### 🚀 How to Use:
1. **Attach your trained model datasets** via **+ Add Input** (you can attach DINO ViT models, ConvNeXt-V2 models, or both!).
2. Ensure the competition dataset (`csiro-biomass`) is attached.
3. Turn on GPU in Notebook Settings (`Accelerator -> GPU T4 or P100`).
4. **Run All** -> Automatically detects models, runs TTA inference, blends families, and outputs `submission.csv` ready to submit.


In [ ]:
# 1. Environment Setup & Offline Support
import os
import sys
import glob
import random
import subprocess
import numpy as np
import pandas as pd
from PIL import Image
import cv2
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Handle timm: works online or from offline .whl if attached
try:
    import timm
except ImportError:
    wheels = glob.glob('/kaggle/input/**/timm*.whl', recursive=True)
    if len(wheels) > 0:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', wheels[0]])
    else:
        subprocess.check_call(['pip', 'install', '-q', 'timm'])
    import timm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('\n' + '!'*60)
    print('[!] CRITICAL WARNING: Running on CPU! Inference will be 50x slower.')
    print('    Please turn on GPU in Notebook Settings (right panel -> Accelerator -> GPU T4 or P100).')
    print('!'*60 + '\n')


In [ ]:
# 2. Configuration & Automatic Path & Checkpoint Discovery
def find_data_dir():
    candidates = [
        '/kaggle/input/competitions/csiro-biomass',
        '/kaggle/input/csiro-biomass',
        '../input/competitions/csiro-biomass',
        '../input/csiro-biomass',
        './data',
        '.'
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, 'test.csv')):
            return c
    if os.path.exists('/kaggle/input'):
        for root, _, files in os.walk('/kaggle/input'):
            if 'test.csv' in files:
                return root
    return '.'

def find_model_checkpoints():
    """Finds all uploaded model checkpoint (.pt or .pth) files across /kaggle/input and working directory."""
    found = []
    if os.path.exists('/kaggle/input'):
        for root, _, files in os.walk('/kaggle/input'):
            # Skip image and source dataset directories
            if any(x in root.lower() for x in ['train', 'test', 'images', 'csiro-biomass/train']):
                continue
            for f in files:
                if f.endswith('.pt') or f.endswith('.pth'):
                    found.append(os.path.join(root, f))
    
    for f in sorted(glob.glob('*.pt') + glob.glob('*.pth')):
        p = os.path.abspath(f)
        if p not in found:
            found.append(p)
            
    model_files = [f for f in found if 'model' in f.lower() or 'fold' in f.lower()]
    return sorted(model_files if model_files else found)

class CFG:
    DATA_DIR = find_data_dir()
    TEST_CSV = os.path.join(DATA_DIR, 'test.csv')
    SAMPLE_SUB_CSV = os.path.join(DATA_DIR, 'sample_submission.csv')
    TEST_IMG_DIR = os.path.join(DATA_DIR, 'test') if os.path.exists(os.path.join(DATA_DIR, 'test')) else DATA_DIR
    
    IMG_SIZE = 512
    FUSION_DIM = 384
    NUM_INTERVALS = 7
    BATCH_SIZE = 8
    USE_TTA = True
    
    # Multi-Backbone Ensemble Blending Weights (Normalized if both present)
    WEIGHT_VIT = 0.50       # Weight for DINOv3 ViT-Base models
    WEIGHT_CONVNEXT = 0.50  # Weight for ConvNeXt-V2 Large models
    
    TARGET_ORDER = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD = [0.229, 0.224, 0.225]

print(f'DATA_DIR: {CFG.DATA_DIR}')
print(f'TEST_CSV: {CFG.TEST_CSV} (Exists: {os.path.exists(CFG.TEST_CSV)})')

CHECKPOINTS = find_model_checkpoints()
print(f'\nDiscovered {len(CHECKPOINTS)} Model Checkpoint(s):')
for cp in CHECKPOINTS:
    print(f'  - {cp} ({os.path.getsize(cp) / (1024*1024):.1f} MB)')

if len(CHECKPOINTS) == 0:
    print('\n[!] Note: No .pt/.pth checkpoints found yet.')
    print('    Please upload your trained models as a Kaggle dataset and attach it via "+ Add Input".')


In [ ]:
# 3. DualStreamBiomassModel Architecture & Smart Architecture Detector
def detect_model_architecture(state_dict):
    """
    Inspects state_dict tensor dimensions and keys to automatically determine
    whether the model is ConvNeXt-V2 Large or DINOv3 ViT-Base.
    """
    if isinstance(state_dict, dict) and 'state_dict' in state_dict:
        if 'backbone' in state_dict:
            return state_dict['backbone'], state_dict['state_dict']
        state_dict = state_dict['state_dict']
        
    clean_sd = {k.replace('module.', ''): v for k, v in state_dict.items()}
    
    # 1. Primary check: cross-view attention projection embed dimension
    if 'cross_view_attn.in_proj_weight' in clean_sd:
        dim = clean_sd['cross_view_attn.in_proj_weight'].shape[1]
        if dim == 1536:
            return 'convnextv2_large', clean_sd
        elif dim == 768:
            return 'vit_base_patch16_dinov3_qkvb', clean_sd
            
    # 2. Secondary check: layer name patterns
    keys = list(clean_sd.keys())
    if any('stages' in k for k in keys) or any('convnext' in k.lower() for k in keys):
        return 'convnextv2_large', clean_sd
        
    return 'vit_base_patch16_dinov3_qkvb', clean_sd

class DualStreamBiomassModel(nn.Module):
    def __init__(self, backbone_name='vit_base_patch16_dinov3_qkvb', num_targets=5, num_intervals=7, fusion_dim=384, dropout=0.3, pretrained=False, **kwargs):
        super().__init__()
        self.backbone_name = backbone_name
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
        self.backbone_dim = self.backbone.num_features
        
        num_heads = 8 if self.backbone_dim % 8 == 0 else 4
        self.cross_view_attn = nn.MultiheadAttention(embed_dim=self.backbone_dim, num_heads=num_heads, dropout=0.1, batch_first=True)
        self.attn_norm = nn.LayerNorm(self.backbone_dim)
        
        self.fusion_mlp = nn.Sequential(
            nn.Linear(self.backbone_dim * 2, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        self.reg_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(fusion_dim, fusion_dim // 2),
                nn.LayerNorm(fusion_dim // 2),
                nn.GELU(),
                nn.Dropout(dropout * 0.5),
                nn.Linear(fusion_dim // 2, 64),
                nn.GELU(),
                nn.Linear(64, 1)
            ) for _ in range(num_targets)
        ])
        
        self.cls_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(fusion_dim, 128),
                nn.LayerNorm(128),
                nn.GELU(),
                nn.Dropout(dropout * 0.5),
                nn.Linear(128, num_intervals)
            ) for _ in range(num_targets)
        ])

    def extract_features(self, x):
        feats = self.backbone(x)
        return feats.mean(dim=1) if len(feats.shape) == 3 else feats.mean(dim=[2, 3]) if len(feats.shape) == 4 else feats

    def forward(self, img_left, img_right):
        feat_l = self.extract_features(img_left)
        feat_r = self.extract_features(img_right)
        
        tokens = torch.stack([feat_l, feat_r], dim=1)
        attn_out, _ = self.cross_view_attn(tokens, tokens, tokens)
        tokens = self.attn_norm(tokens + attn_out)
        
        fused = self.fusion_mlp(torch.cat([tokens[:, 0], tokens[:, 1]], dim=-1))
        reg_preds = [F.softplus(head(fused)) for head in self.reg_heads]
        cls_preds = [head(fused) for head in self.cls_heads]
        return reg_preds, cls_preds


In [ ]:
# 4. Dual-Stream Test Dataset & Image Resolver
class DualStreamTestDataset(Dataset):
    def __init__(self, df, img_dir=CFG.TEST_IMG_DIR, img_size=CFG.IMG_SIZE):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.img_size = img_size
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=CFG.IMAGENET_MEAN, std=CFG.IMAGENET_STD),
        ])

    def __len__(self):
        return len(self.df)

    def _resolve_image_path(self, raw_path):
        if os.path.exists(raw_path):
            return raw_path
        fname = os.path.basename(raw_path)
        candidates = [
            os.path.join(CFG.DATA_DIR, raw_path),
            os.path.join(CFG.DATA_DIR, 'test', fname),
            os.path.join(CFG.DATA_DIR, 'train', fname),
            os.path.join(self.img_dir, fname) if self.img_dir else None,
            os.path.join(self.img_dir, raw_path) if self.img_dir else None,
            os.path.join('/kaggle/input/competitions/csiro-biomass', raw_path),
            os.path.join('/kaggle/input/csiro-biomass', raw_path),
            os.path.join('test', fname),
            os.path.join('train', fname)
        ]
        for c in candidates:
            if c and os.path.exists(c):
                return c
        return raw_path

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = self._resolve_image_path(row['image_path'])
        img = cv2.imread(path)
        if img is None:
            img = np.zeros((1000, 2000, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
        h, w, _ = img.shape
        mid = w // 2
        left_np = img[:, :mid, :]
        right_np = img[:, mid:, :]
        
        tensor_l = self.transform(Image.fromarray(left_np))
        tensor_r = self.transform(Image.fromarray(right_np))
        
        return {
            'image_left': tensor_l,
            'image_right': tensor_r,
            'sample_id': row.get('sample_id', row.get('clean_id', f'sample_{idx}')),
        }


In [ ]:
# 5. Soft Physical Post-Processing & Calibration
def soft_physics_postprocess(preds_np):
    """Enforces physical mass relationships and thatch fringe expansion:"""
    preds = np.maximum(preds_np.copy(), 0.0)
    green = preds[:, 0]
    dead = preds[:, 1]
    clover = preds[:, 2] * 0.8
    gdm = preds[:, 3]
    total = preds[:, 4]
    
    # 3rd-Place Dead Thatch Fringe Expansion
    dead = np.where(dead > 20.0, dead * 1.1, np.where(dead < 10.0, dead * 0.9, dead))
    
    # Mass-conservation blends
    gdm_blended = 0.5 * gdm + 0.5 * (green + clover)
    total_blended = 0.5 * total + 0.5 * (green + clover + dead)
    
    return np.maximum(np.column_stack([green, dead, clover, gdm_blended, total_blended]), 0.0)


In [ ]:
# 6. Multi-Backbone Super-Ensemble Inference with TTA
assert len(CHECKPOINTS) > 0, 'ERROR: No checkpoints found! Please attach your trained models dataset to the notebook.'

# 1. Read Test Data
test_df_raw = pd.read_csv(CFG.TEST_CSV)
if 'target_name' in test_df_raw.columns:
    test_df_raw['clean_id'] = test_df_raw['sample_id'].astype(str).apply(lambda x: x.split('__')[0])
    unique_test = test_df_raw[['clean_id', 'image_path']].drop_duplicates(subset=['clean_id']).reset_index(drop=True)
else:
    unique_test = test_df_raw.copy()
    if 'clean_id' not in unique_test.columns:
        unique_test['clean_id'] = unique_test['sample_id']

print(f'Test set: {len(unique_test)} unique pasture images to predict.')

# 2. Create DataLoader
test_dataset = DualStreamTestDataset(unique_test, CFG.TEST_IMG_DIR, CFG.IMG_SIZE)
test_loader = DataLoader(test_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=0)

# 3. Predict across all discovered model checkpoints, grouped by architecture
vit_predictions = []
convnext_predictions = []
all_model_info = []

for cp_idx, cp_path in enumerate(CHECKPOINTS):
    cp_name = os.path.basename(cp_path)
    raw_state = torch.load(cp_path, map_location=DEVICE)
    backbone_name, clean_sd = detect_model_architecture(raw_state)
    
    family = 'ConvNeXt' if 'convnext' in backbone_name else 'DINO ViT'
    print(f'\n[{cp_idx + 1}/{len(CHECKPOINTS)}] Evaluating: {cp_name} | Family: {family} ({backbone_name})...')
    all_model_info.append({'path': cp_path, 'name': cp_name, 'backbone': backbone_name, 'family': family})
    
    model = DualStreamBiomassModel(backbone_name=backbone_name, pretrained=False).to(DEVICE)
    model.load_state_dict(clean_sd)
    model.eval()
    
    fold_preds = []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f'Inference {cp_name}', leave=False):
            img_l = batch['image_left'].to(DEVICE)
            img_r = batch['image_right'].to(DEVICE)
            
            if CFG.USE_TTA:
                # Standard view
                r_std, _ = model(img_l, img_r)
                # Mirrored horizontal view (flip right view as left, flip left view as right)
                r_flip, _ = model(torch.flip(img_r, [3]), torch.flip(img_l, [3]))
                avg_r = [(a + b) * 0.5 for a, b in zip(r_std, r_flip)]
            else:
                avg_r, _ = model(img_l, img_r)
                
            pred_batch = torch.cat(avg_r, dim=1).cpu().numpy()
            fold_preds.append(pred_batch)
            
    fold_preds_arr = np.concatenate(fold_preds, axis=0)
    if family == 'ConvNeXt':
        convnext_predictions.append(fold_preds_arr)
    else:
        vit_predictions.append(fold_preds_arr)

# 4. Multi-Backbone Blended Ensembling
print(f'\n======================================================')
print(f'Ensemble Summary:')
print(f'  - DINO ViT Models:     {len(vit_predictions)}')
print(f'  - ConvNeXt-V2 Models:  {len(convnext_predictions)}')
print(f'======================================================')

if len(vit_predictions) > 0 and len(convnext_predictions) > 0:
    vit_raw = np.mean(vit_predictions, axis=0)
    convnext_raw = np.mean(convnext_predictions, axis=0)
    
    # Normalize blend weights
    w_total = CFG.WEIGHT_VIT + CFG.WEIGHT_CONVNEXT
    w_v = CFG.WEIGHT_VIT / w_total
    w_c = CFG.WEIGHT_CONVNEXT / w_total
    
    ensemble_raw = w_v * vit_raw + w_c * convnext_raw
    print(f'✓ Multi-Backbone Blend Applied: {w_v:.0%} ViT + {w_c:.0%} ConvNeXt-V2')
elif len(vit_predictions) > 0:
    ensemble_raw = np.mean(vit_predictions, axis=0)
    print(f'✓ ViT-Only Ensemble Applied ({len(vit_predictions)} models)')
else:
    ensemble_raw = np.mean(convnext_predictions, axis=0)
    print(f'✓ ConvNeXt-Only Ensemble Applied ({len(convnext_predictions)} models)')

ensemble_post = soft_physics_postprocess(ensemble_raw)


In [ ]:
# 7. Generate, Verify & Export Submissions
clean_ids = unique_test['clean_id'].tolist()

def make_submission(preds_post, filename):
    pred_dict = {
        clean_ids[i]: {col: float(preds_post[i, c_idx]) for c_idx, col in enumerate(CFG.TARGET_ORDER)}
        for i in range(len(clean_ids))
    }
    
    if 'target_name' in test_df_raw.columns:
        sub_df = test_df_raw.copy()
        sub_df['target'] = sub_df.apply(
            lambda row: pred_dict.get(row['clean_id'], {}).get(row['target_name'], 0.0), axis=1
        )
        res = sub_df[['sample_id', 'target']].copy()
    elif os.path.exists(CFG.SAMPLE_SUB_CSV):
        sub_t = pd.read_csv(CFG.SAMPLE_SUB_CSV)
        sub_t['clean_id'] = sub_t['sample_id'].astype(str).apply(lambda x: x.split('__')[0])
        sub_t['target_name'] = sub_t['sample_id'].astype(str).apply(lambda x: x.split('__')[1])
        sub_t['target'] = sub_t.apply(
            lambda row: pred_dict.get(row['clean_id'], {}).get(row['target_name'], 0.0), axis=1
        )
        res = sub_t[['sample_id', 'target']].copy()
    else:
        records = []
        for cid in clean_ids:
            for col in CFG.TARGET_ORDER:
                records.append({'sample_id': f'{cid}__{col}', 'target': pred_dict[cid][col]})
        res = pd.DataFrame(records)
        
    assert res['target'].isna().sum() == 0, f'ERROR: {filename} contains NaN values!'
    assert len(res) > 0, f'ERROR: {filename} is empty!'
    res.to_csv(filename, index=False)
    return res

# Primary blended submission (evaluated by Kaggle)
final_sub = make_submission(ensemble_post, 'submission.csv')

# Individual family auxiliary submissions if both available
if len(vit_predictions) > 0 and len(convnext_predictions) > 0:
    make_submission(soft_physics_postprocess(np.mean(vit_predictions, axis=0)), 'submission_vit.csv')
    make_submission(soft_physics_postprocess(np.mean(convnext_predictions, axis=0)), 'submission_convnextv2.csv')
    print('Saved auxiliary submissions: submission_vit.csv and submission_convnextv2.csv')

print('=' * 50)
print(f'SUCCESS! Primary submission saved to: submission.csv')
print(f'Total predictions: {len(final_sub)}')
print('=' * 50)
print('\nTarget Distribution:')
print(final_sub['target'].describe())
print('\nFirst 10 Rows:')
print(final_sub.head(10))
